In [1]:
using CommonDataFormat
using Dates, CSV, DataFrames
using Statistics
using CairoMakie
include("../../src/scripts/TW_load.jl")
include("../../src/scripts/wave_caculate.jl")
include("../../src/scripts/MAVEN_load.jl")
include("../../src/scripts/MAVEN_plot.jl")
include("../../src/scripts/MAVEN_SWIA.jl")
# include("../../src/scripts/MAVEN_STATIC.jl")
using .TW_load
using .WaveCaculate
using .MAVEN_load
using .MAVEN_load
using .MAVEN_plot
using .MAVEN_SWIA
# using .MAVEN_STATIC

date = Date(2022, 8, 24)
time_range = DateTime(date, Time(7, 00, 00)) .+ Dates.Minute(20) .* range(0, 6)
date_str = Dates.format.(date, "yyyymmdd")
date_str_dye = "$(year(date))$(lpad(dayofyear(date), 3, '0'))"
# dir = pwd()
dir = joinpath(@__DIR__, "..", "..")
input_path = joinpath(dir, "Data", "MAVEN")
# save_path = joinpath(dir, "results", "MAVEN", save_name)
data_path = joinpath(input_path, "mvn_swi_l2_coarsesvy3d_$(date_str)_v02_r01.cdf")
quat_path = joinpath(input_path, "mvn_spice_swia_qu_$(date_str).csv")
mag_path = joinpath(input_path, "mvn_mag_l3_$(date_str_dye)ss1s_$(date_str)_v01_r01.f77_unformatted")

data = MAVEN_load.load_cdf(data_path)
quat_data = MAVEN_load.load_quat(quat_path)
quat_data[:data_load_flag] = true
swi_data = MAVEN_SWIA.get_3dc!(data, quat_data = quat_data)
mag_data = MAVEN_load.load_mag_l3(mag_path)
println("Data loaded.")

Data loaded.


In [ ]:
time_slice = DateTime(date, Time(7, 30, 00))
swi_idx = argmin(abs.(swi_data[:epoch] .- time_slice))
flux_4d = swi_data[:diff_en_fluxes][swi_idx, :, :, :]
flux_phi = Matrix(dropdims(mean(flux_4d, dims = 2), dims = 2)')
flux_theta = Matrix(dropdims(mean(flux_4d, dims = 1), dims = 1)')
energies = vec(swi_data[:energy])
phi = swi_data[:phi]

In [ ]:
using CairoMakie
CairoMakie.activate!()
set_theme!(;
    Axis=(xscale = log10, yscale = log10,
        bordercolor=:black, bordersize=15, spinewidth = 2, 
        xlabelsize=18, ylabelsize=18, xticklabelsize=16, yticklabelsize=16, 
        xminorticksvisible = true, yminorticksvisible = true,
        xminorticks = IntervalsBetween(10), yminorticks = IntervalsBetween(10))
)
fig = Figure(resolution = (800, 1200))
ax1 = Axis(fig[1, 1];xlabel="Energy (eV)" ,ylabel="Energy Flux")
ax2 = Axis(fig[2, 1];xlabel="Energy (eV)" ,ylabel="Energy Flux")
colors = cgrad(:viridis, 16, categorical = true)
ylims!(ax1, 1e4, 1e9)
for i in 1:16
lines!(ax1, energies, flux_phi[:, i], color = colors[i], label = "$i")
end
# 添加图例
Legend(fig[1, 2], ax1, "Channel", nbanks = 2)
# --- 绘制下图 (示例：只选前 4 个通道) ---
for i in 1:4
lines!(ax2, energies, flux_theta[:, i], color = colors[i], label = "$i", linewidth = 2)
end
Legend(fig[2, 2], ax2, "Channel")
display(fig)

In [ ]:
time_range = DateTime(date, Time(6, 30, 00)) .+ Dates.Minute(20) .* range(0, 6)

In [ ]:
time_range = DateTime(date, Time(6, 30, 00)) .+ Dates.Minute(20) .* range(0, 6)
energy_range = [4.0, 10.0] # keV

using CairoMakie
CairoMakie.activate!()

fig = Figure(size = (1600, 400))
ax = Axis(fig[1,1])

hm = swi_pitch_angle(ax, swi_data, mag_data, time_range, energy_range)

xtk = datetime2julian.(time_range)
xlims!(ax, xtk[1], xtk[end])
ax.xticks = (xtk, Dates.format.(time_range, "HH:MM:SS"))
ax.xlabel = "Time Index"
ax.ylabel = "Pitch Angle (degrees)"
Colorbar(fig[1,2], hm, label="log₁₀(Flux)")

save(save_path, fig)

In [ ]:
time_range = DateTime(date, Time(6, 30, 00)) .+ Dates.Minute(20) .* range(0, 6)
energy_range = [4, 10] # keV
time_idx_mag = findall(minimum(time_range).<mag_data[:epoch].<maximum(time_range))
mag_epochs = mag_data[:epoch][time_idx_mag]
energies = swi_data[:energy][:] ./ 1000 
pitch_angle = swi_data[:phi][:]
time_idx = findall(minimum(time_range).<swi_data[:epoch].<maximum(time_range))
energy_idx = findall(minimum(energy_range).<energies.<maximum(energy_range)) 
ntime = length(time_idx)
nenergy = length(energy_idx)
ntheta = 4
nphi = 16
swi_epochs = swi_data[:epoch][time_idx]
flux_4d = swi_data[:diff_en_fluxes][time_idx, :, :, energy_idx]
pitch_angle_4d = Array{Float64, 4}(undef, ntime, nphi, ntheta, nenergy)
for i in 1:ntime
    energy = reshape(swi_data[:energy][energy_idx],1,1,nenergy)
    phi = reshape(swi_data[:phi][:],nphi,1,1)
    theta = reshape(swi_data[:theta][time_idx[i], :, energy_idx],1,ntheta,nenergy)
    v0 = MAVEN_SWIA.ion_energy2v.(energy,1)
    vv = MAVEN_SWIA.sphere2xyz_for_SWIA.(v0,theta,phi)
    mag_idx = argmin(abs.(mag_epochs .- swi_epochs[i]))
    B = convert(Vector{Float64},mag_data[:B][mag_idx, :])
    pitch_angle_4d[i, :, :, :] = angle_between.(vv, Ref(B))
end

pitch_angle_2d = reshape(pitch_angle_4d, ntime, nphi*ntheta*nenergy)
flux_2d = reshape(flux_4d, ntime, nphi*ntheta*nenergy)

# 创建 pitch angle bins (0-180度)
n_bins = 60
bin_edges = range(0, 180, length=n_bins+1)
bin_centers = (bin_edges[1:end-1] + bin_edges[2:end]) / 2
# 构建热力图矩阵
heatmap_data = zeros(Float64, ntime, n_bins)
for t in 1:ntime
    pitch_angle_t = pitch_angle_2d[t, :]
    flux_t = flux_2d[t, :]
    
    for i in 1:(nphi*ntheta*nenergy)
        pa_val = pitch_angle_t[i]
        flux_val = flux_t[i]
        
        # 找到对应的 bin
        bin_idx = floor(Int, pa_val / 180 * n_bins) + 1
        bin_idx = clamp(bin_idx, 1, n_bins)
        
        heatmap_data[t, bin_idx] += flux_val
    end
end

In [ ]:
using CairoMakie
CairoMakie.activate!()

fig = Figure(size = (1600, 400))
ax = Axis(fig[1,1])
xtk = datetime2julian.(time_range)
threshold = 1e2  # 你想设定的阈值
# 将小于阈值的值替换为 NaN
heatmap_data_masked = ifelse.(heatmap_data .< threshold, NaN, heatmap_data)
hm = heatmap!(ax, xtk, range(0, 180, length=n_bins), heatmap_data_masked, 
    colormap=cgrad(:RdYlBu, rev=true), colorscale=log10, colorrange = (1e5, 1e8))

xlims!(ax, xtk[1], xtk[end])
ax.xticks = (xtk, Dates.format.(time_range, "HH:MM:SS"))
ax.xlabel = "Time Index"
ax.ylabel = "Pitch Angle (degrees)"
Colorbar(fig[1,2], hm, label="log₁₀(Flux)")

save(save_path, fig)

In [ ]:
maximum(log_heatmap)

In [ ]:
using CairoMakie
CairoMakie.activate!()

fig = Figure()
ax = Axis(fig[1,1])
heatmap!(ax, 1:ntime, range(0, 180, length=n_bins), log10.(max.(heatmap_data, 1e-10)))
xtk = datetime2julian.(time_range)
xlims!(ax, xtk[1], xtk[end])
ax.xticks = (xtk, Dates.format.(time_range, "HH:MM:SS"))
ax.xlabel = "Time Index"
ax.ylabel = "Pitch Angle (degrees)"
Colorbar(fig[1,2], label="log₁₀(Flux)")

save(save_path, fig)

In [ ]:
save_path

In [ ]:
heatmap_data

In [ ]:
all_pitch_flat = vec(pitch_angle_4d)

In [ ]:
pitch_angle_4d

In [ ]:
flux_4d

In [ ]:
theta[1,:,:]

In [ ]:
theta_perm = permutedims(theta, (1, 3, 2))
v0 = MAVEN_SWIA.ion_energy2v.(energy,1)
vv = MAVEN_SWIA.sphere2xyz_for_SWIA.(v0,theta_perm,phi)
velocity = permutedims(vv, (1, 3, 2, 4))

In [ ]:
swi_epochs

In [ ]:
time_idx_mag = findall(minimum(time_range).<mag_data[:epoch].<maximum(time_range))
mag_epochs = mag_data[:epoch][time_idx_mag]
# B = mag_data[:B][time_idx_mag, :]

In [ ]:
flux_4d

In [ ]:
mag_data

In [ ]:
swi_data

In [ ]:
swi_data[:energy_coarse]

In [ ]:
swi_data[:diff_en_fluxes]

In [ ]:
swi_data[:theta]


In [ ]:
swi_data[:phi]